In [ ]:
# Here I will perform the same models as in the previous file
# rather than extracting text statistics, I will use Tf Idf Vectorization

import kagglehub
import os
import pandas as pd


path = kagglehub.dataset_download("shanegerami/ai-vs-human-text")
print(" Dataset downloaded to:", path)

csv_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_path = os.path.join(root, f)

if csv_path is None:
    raise FileNotFoundError("CSV not found!")

print("Found CSV:", csv_path)

df = pd.read_csv(csv_path)

print("\nDataset loaded")
print(df.head())
print("\nClass distribution:  0 (Human), 1(AI generated)")
print(df["generated"].value_counts())

#Balancing the dataset

ai_written=df[df["generated"]== 1].sample(5000, random_state=42)
human_written = df[df["generated"] == 0].sample(5000, random_state=42)

balanced_dataset=pd.concat([ai_written, human_written]).sample(frac=1,random_state=42)

print(balanced_dataset["generated"].value_counts())
print(balanced_dataset.head())



 Dataset downloaded to: C:\Users\xhani\.cache\kagglehub\datasets\shanegerami\ai-vs-human-text\versions\1
Found CSV: C:\Users\xhani\.cache\kagglehub\datasets\shanegerami\ai-vs-human-text\versions\1\AI_Human.csv

Dataset loaded
                                                text  generated
0  Cars. Cars have been around since they became ...        0.0
1  Transportation is a large necessity in most co...        0.0
2  "America's love affair with it's vehicles seem...        0.0
3  How often do you ride in a car? Do you drive a...        0.0
4  Cars are a wonderful thing. They are perhaps o...        0.0

Class distribution:  0 (Human), 1(AI generated)
generated
0.0    305797
1.0    181438
Name: count, dtype: int64
generated
0.0    5000
1.0    5000
Name: count, dtype: int64
                                                     text  generated
269265  Dear Principal,\n\nWe have been hearing quite ...        0.0
327273   Dear [State Senator],\n\nI am writing to expr...        1.0
167398  As

In [3]:
from sklearn.model_selection import train_test_split

X=balanced_dataset["text"]
y=balanced_dataset["generated"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [4]:
#TF IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec=TfidfVectorizer(max_features=20000,ngram_range=(1,2),stop_words="english")
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape :", X_test_tfidf.shape)

TF-IDF train shape: (8000, 20000)
TF-IDF test shape : (2000, 20000)


In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
dt_tfidf = DecisionTreeClassifier(
    max_depth=10,criterion='gini',min_samples_leaf=10,
    random_state=42
)

dt_tfidf.fit(X_train_tfidf, y_train)

preds = dt_tfidf.predict(X_test_tfidf)


print("DT Accuracy :", round(accuracy_score(y_test, preds),4))


DT Accuracy : 0.85


In [28]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC(kernel='rbf',C=10,gamma='scale')
svm.fit(X_train_tfidf, y_train)

preds = svm.predict(X_test_tfidf)

print("SVM Accuracy:", round(accuracy_score(y_test, preds),4))


SVM Accuracy: 0.981


In [29]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

svm = LinearSVC(C=10)
svm.fit(X_train_tfidf, y_train)

preds = svm.predict(X_test_tfidf)

print("Linear SVM Accuracy:", round(accuracy_score(y_test, preds),4))


Linear SVM Accuracy: 0.986


In [30]:
from sklearn.ensemble import RandomForestClassifier

forest=RandomForestClassifier(n_estimators=100,criterion='gini',max_depth=None,min_samples_leaf=1)
forest.fit(X_train_tfidf,y_train)
rf_preds=forest.predict(X_test_tfidf)
rf_acc=accuracy_score(y_test,rf_preds)

print("random forest accuracy: ",rf_acc)

random forest accuracy:  0.966
